# **Проект: Исследование надежности заемщиков**

## **О проекте**

### **Описание проекта**

Заказчик — кредитный отдел банка. Нужно разобраться, влияет ли семейное положение и количество детей клиента на факт погашения кредита в срок. Входные данные от банка — статистика о платёжеспособности клиентов.

Результаты исследования будут учтены при построении модели кредитного скоринга — специальной системы, которая оценивает способность потенциального заёмщика вернуть кредит банку.

**Пояснения к проекту**

*Первая часть является выполнением цепочки заранее зафиксированных задач, проверявшихся автоматически (поэтому ход работы может быть не совсем корректен).*

*Вторая же часть давала больше самостоятельности и получала фидбек уже от живого человека.*

### **Описание данных**

Получены данные от заказчика, в которых описаны различные параметры их клиентов-кредитополучателей.

## **Часть 1: Подготовка данных**

### **Загрузка данных и изучение общей информации**

 Импортирую библиотеку pandas. Считаю данные из csv-файла в датафрейм и сохраню в переменную `'data'`.

In [1]:
# загрузка необходимой библиотеки
import pandas as pd

In [ ]:
# загрузка данных
try:
    data = pd.read_csv('/datasets/data.csv')
except:
  # ссылка искажена для этичности работы с данными
    data = pd.read_csv('https://ссылка_удалена/datasets/data.csv')

Выведу первые 20 строчек датафрейма на экран.

In [3]:
data.head(20) # вывод первых строк

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу
5,0,-926.185831,27,высшее,0,гражданский брак,1,M,компаньон,0,255763.565419,покупка жилья
6,0,-2879.202052,43,высшее,0,женат / замужем,0,F,компаньон,0,240525.971920,операции с жильем
7,0,-152.779569,50,СРЕДНЕЕ,1,женат / замужем,0,M,сотрудник,0,135823.934197,образование
8,2,-6929.865299,35,ВЫСШЕЕ,0,гражданский брак,1,F,сотрудник,0,95856.832424,на проведение свадьбы
9,0,-2188.756445,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425.938277,покупка жилья для семьи


Выведу основную информацию о датафрейме.

In [4]:
data.info() # вывод основной информации

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


### **Предобработка данных**

#### **Удаление пропусков**

Выведу количество пропущенных значений для каждого столбца используя комбинацию двух методов.

In [5]:
data.isna().sum() # пропущенные значения для каждого столбца

,0
children,0
days_employed,2174
dob_years,0
education,0
education_id,0
family_status,0
family_status_id,0
gender,0
income_type,0
debt,0


В двух столбцах есть пропущенные значения. Один из них — `'days_employed'`. Пропуски в этом столбце я обработаю на следующем этапе. Другой столбец с пропущенными значениями — `'total_income'` — хранит данные о доходах. На сумму дохода сильнее всего влияет тип занятости, поэтому заполнить пропуски в этом столбце нужно медианным значением по каждому типу из столбца `'income_type'`. Например, у человека с типом занятости сотрудник пропуск в столбце `'total_income'` должен быть заполнен медианным доходом среди всех записей с тем же типом.

*Потенциально, лучше было бы сначала проверить данные на аномальность в данных столбцах, а потом заменять медианой. Однако в условиях выполняемого задания требовалось именно так.*

In [6]:
# замена пропусков числовых значений медианой
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['total_income'].isna()), 'total_income'] = \
    data.loc[(data['income_type'] == t), 'total_income'].median()

#### **Обработка аномальных значений**

 В данных могут встречаться артефакты (аномалии) — значения, которые не отражают действительность и появились по какой-то ошибке. Таким артефактом будет, например, отрицательное количество дней трудового стажа в столбце `'days_employed'`. Для реальных данных это нормально.

 Обработаю значения в этом столбце: заменю все отрицательные значения положительными.


In [7]:
# замена отрицательных значений
data['days_employed'] = data['days_employed'].abs()

Для каждого типа занятости выведу медианное значение трудового стажа `'days_employed'` в днях.

In [8]:
# считаем медиану
data.groupby('income_type')['days_employed'].agg('median')

,days_employed
income_type,
безработный,366413.652744
в декрете,3296.759962
госслужащий,2689.368353
компаньон,1547.382223
пенсионер,365213.306266
предприниматель,520.848083
сотрудник,1574.202821
студент,578.751554


У двух типов (безработные и пенсионеры) получаюся аномально большие значения. Исправить такие значения сложно, поэтому придется оставьте их как есть. Тем более этот столбец (согласно заданиям) не понадобится для дальнейших исследований.

Выведу перечень уникальных значений столбца `'children'`.

In [9]:
# уникальные значения
data['children'].unique()

array([ 1,  0,  3,  2, -1,  4, 20,  5])

В столбце `'children'` есть два аномальных значения. Необходимо будет удалить строки, в которых встречаются такие аномальные значения из датафрейма data.

Возможно, -1 на самом деле 1, а 20 равно 2. Можно уточнить эти сведения у заемщиков или из иных баз данных. Но это не слишком рациональная трата сил, так как потеря нескольких строк с такими артефактами не окажет значительного влияния на результаты исследования. То есть польза от нахождения "правильных данных" значительно уступает затраченным на их нахождение усилиям.

Поэтому данные аномалии просто удаляем.

In [10]:
# удаляю аномальные строки
data = data[(data['children'] != -1) & (data['children'] != 20)]

Ещё раз выведу перечень уникальных значений столбца `'children'`, чтобы убедиться, что артефакты удалены.

In [11]:
# уникальные значения
data['children'].unique()

array([1, 0, 3, 2, 4, 5])

#### **Удаление пропусков (продолжение)**

Заполню пропуски в столбце `'days_employed'` медианными значениями по каждого типа занятости `'income_type'`.

In [12]:
# заменяю медианой
for d in data['income_type'].unique():
    data.loc[(data['income_type'] == d)&(data['days_employed'].isna()),'days_employed'] = \
    data.loc[(data['income_type'] == d), 'days_employed'].median()

Убежусь, что все пропуски заполнены. Для этого ещё раз выведу количество пропущенных значений для каждого столбца с помощью двух методов.

In [13]:
data.isna().sum() # пропущенные значения для каждого столбца

,0
children,0
days_employed,0
dob_years,0
education,0
education_id,0
family_status,0
family_status_id,0
gender,0
income_type,0
debt,0


#### **Изменение типов данных**

Заменю вещественный тип данных в столбце `'total_income'` на целочисленный.

In [14]:
# замена типа данных
data['total_income'] = data['total_income'].astype(int)

*Тип данных в стобце 'total_income' - float64, а пропуски были успешно удалены ранее, поэтому можно обойтись без проверки успешности замены.*

#### **Обработка дубликатов**

Обработаю неявные дубликаты в столбце `'education'`. В этом столбце есть одни и те же значения, но записанные по-разному: с использованием заглавных и строчных букв. Приведу их к нижнему регистру, а также проверю остальные столбцы.

In [15]:
# привожу к нижнему регистру
data['education'] = data['education'].str.lower()

Выведу на экран количество строк-дубликатов в данных. Если такие строки присутствуют, удалю их.

In [16]:
# считаю дубликаты
data.duplicated().sum()

np.int64(71)

In [17]:
# удаляю дубликаты
data = data.drop_duplicates()

In [18]:
# проверка на отсутствие явных дубликатов
data.duplicated().sum()

np.int64(0)

#### **Категоризация данных**

На основании диапазонов, указанных ниже, необходимо в датафрейме `'data'` столбец `'total_income_category'` с категориями:

- 0–30000 — 'E';
- 30001–50000 — 'D';
- 50001–200000 — 'C';
- 200001–1000000 — 'B';
- 1000001 и выше — 'A'.

Например, кредитополучателю с доходом 25000 нужно назначить категорию 'E', а клиенту, получающему 235000, — 'B'. Для этого использую собственную функцию с именем `'categorize_income()'` и метод apply().

In [19]:
# создаю функцию
def categorize_income(income):
    try:
        if 0 <= income <= 30000:
            return 'E'
        elif 30001 <= income <= 50000:
            return 'D'
        elif 50001 <= income <= 200000:
            return 'C'
        elif 200001 <= income <= 1000000:
            return 'B'
        elif income >= 1000001:
            return 'A'
    except:
        pass

In [20]:
# применяю ее
data['total_income_category'] = data['total_income'].apply(categorize_income)

Выведу на экран перечень уникальных целей взятия кредита из столбца `'purpose'`.

In [21]:
# уникальные значения
data['purpose'].unique()

array(['покупка жилья', 'приобретение автомобиля',
       'дополнительное образование', 'сыграть свадьбу',
       'операции с жильем', 'образование', 'на проведение свадьбы',
       'покупка жилья для семьи', 'покупка недвижимости',
       'покупка коммерческой недвижимости', 'покупка жилой недвижимости',
       'строительство собственной недвижимости', 'недвижимость',
       'строительство недвижимости', 'на покупку подержанного автомобиля',
       'на покупку своего автомобиля',
       'операции с коммерческой недвижимостью',
       'строительство жилой недвижимости', 'жилье',
       'операции со своей недвижимостью', 'автомобили',
       'заняться образованием', 'сделка с подержанным автомобилем',
       'получение образования', 'автомобиль', 'свадьба',
       'получение дополнительного образования', 'покупка своего жилья',
       'операции с недвижимостью', 'получение высшего образования',
       'свой автомобиль', 'сделка с автомобилем',
       'профильное образование', 'высшее об

*Судя по разнообразию вариантов у одной и той же категории целей - цель покупки заемщики вписывали сами. Возможно, для будущего упрощения обработки информации в анкете стоит предлагать выбрать из нескольких вариантов целей вместо самостоятельного описания ее заемщиками (это также снизит нагрузку по интерпретации при занесении в базу на операционистов).*

Создам функцию, которая на основании данных из столбца `'purpose'` сформирует новый столбец `'purpose_category'`, в который войдут следующие категории:**

- `'операции с автомобилем'`,
- `'операции с недвижимостью'`,
- `'проведение свадьбы'`,
- `'получение образования'`.

Например, если в столбце `'purpose'` находится подстрока `'на покупку автомобиля'`, то в столбце `'purpose_category'` должна появиться строка `'операции с автомобилем'`.

Для этого использую собственную функцию с именем `'categorize_purpose()'` и метод `'apply()'`. И изучу данные в столбце `'purpose'` и определю, какие подстроки помогут мне правильно определить категорию.

In [22]:
# создание функции
def categorize_purpose(row):
    try:
        if 'автом' in row:
            return 'операции с автомобилем'
        elif 'жил' in row or 'недвиж' in row:
            return 'операции с недвижимостью'
        elif 'свад' in row:
            return 'проведение свадьбы'
        elif 'образов' in row:
            return 'получение образования'
    except:
        return 'нет категории'

In [23]:
# использование функции
data['purpose_category'] = data['purpose'].apply(categorize_purpose)

In [24]:
data['purpose_category'].unique() # проверка на успешность категоризации

array(['операции с недвижимостью', 'операции с автомобилем',
       'получение образования', 'проведение свадьбы'], dtype=object)

## **Часть 2: Исследование данных и ответы на вопросы**

### **Вопрос 1: Есть ли зависимость между количеством детей и возвратом кредита в срок?**

Найду, какую долю составляют должники в зависимости от количества у них детей. Для этого создадам новую таблицу со сгруппированными по количеству детей данными о задолженностях, а результат расчетов выведу в столбце `'share_debt'`, для удобства восприятия отсортированном по возрастанию.


In [25]:
# создаю новую таблицу
children_grouped = data.groupby('children').agg({'debt':['count', 'sum']})

In [26]:
# считаю в новый столбец
children_grouped['share_debt'] = children_grouped['debt']['sum']/children_grouped['debt']['count']

In [27]:
# группирую вывод
children_grouped.sort_values(by='share_debt')

debt       share_debt
          count   sum           
children                        
5             9     0   0.000000
0         14091  1063   0.075438
3           330    27   0.081818
1          4808   444   0.092346
2          2052   194   0.094542
4            41     4   0.097561

In [28]:
# альтернативный (сокращенный) вариант, для перепроверки
data.pivot_table(index=['children'], values = ['debt']).rename(columns={'debt':'share_debt'}).sort_values(by='share_debt')

,share_debt
children,
5,0.000000
0,0.075438
3,0.081818
1,0.092346
2,0.094542
4,0.097561


Видно, что в целом семьи с детьми менее надежные заемщики, чем семьи без детей. Но линейной связи между количеством детей и возвратом кредита в срок нет.

Так как семей, имеющих три и более ребенка заметно меньше (количество семей с пятью детьми и вовсе на грани погрешности), чем не имеющих либо имеющих одного или двух, объединим таких заемщиков в отдельную категорию - многодетные. Доли должников среди семей с одним либо двумя детьми весьма близки между собой, так что их также можно объединить в одну группу.

Повторю расчеты. Чтобы снизить вычислительную нагрузку (не обрабатывать весь датафрейм), создадам для этого на основе датафрейма `'data'` новую таблицу `child_debt` с интересующими нас столбцами `'children'`и `'debt'`.

In [29]:
child_debt = data[['children', 'debt']] # создаем таблицу

In [30]:
child_debt.head() # проверяем, что создание таблицы прошло успешно

,children,debt
0,1,0
1,1,0
2,0,0
3,3,0
4,0,0


In [31]:
# создание функции
def categorize_child(children):
    try:
        if children >=3 :
            return 'многодетный'
        elif children == 1 or children == 2:
            return 'один или два ребенка'
        else:
            return 'нет детей'
    except:
        pass

In [32]:
# применяю
# ругается, но работает
child_debt['categorize_child'] = child_debt['children'].apply(categorize_child)

/tmp/ipython-input-2923249077.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  child_debt['categorize_child'] = child_debt['children'].apply(categorize_child)


In [33]:
child_debt.head() #проверяем, что проблемный код все-таки сработал

,children,debt,categorize_child
0,1,0,один или два ребенка
1,1,0,один или два ребенка
2,0,0,нет детей
3,3,0,многодетный
4,0,0,нет детей


Так как нас интересует только итоговый столбец `'share_debt'`, использую метод `'pivot_table()'`

In [34]:
# свожу
child_debt.pivot_table(index=['categorize_child'], values = ['debt'])\
.rename(columns={'debt':'share_debt'}).sort_values(by='share_debt')

,share_debt
categorize_child,
нет детей,0.075438
многодетный,0.081579
один или два ребенка,0.093003


Выходит, что многодетные семьи более добросовестные заемщики, чем семьи с одним либо двумя детьми, но все равно менее, чем семьи без детей.

Вероятно такой результат вызван тем, что один либо два ребенка более распространенное количество детей в семье, чем три и более, а семья с детьми имеет больше статей расходов (куда могут быть затрачены средства, предназначенные для покрытия кредита), особенно непредвиденных, чем семья без детей.

Также можно дополнительно изучить наличие зависимости между наличием (независимо от количества) или отсутствием детей и возвратом кредита в срок.

In [35]:
# создаю функцию
def children_have(children):
    try:
        if children == 0:
            return 'без детей'
        else:
            return 'есть дети'
    except:
        pass

In [36]:
# применяю
# ругается, но работает
child_debt['child_have'] = child_debt['children'].apply(children_have)

/tmp/ipython-input-1703491820.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  child_debt['child_have'] = child_debt['children'].apply(children_have)


In [37]:
child_debt.head() # проверяем, что проблемный код все-таки сработал

,children,debt,categorize_child,child_have
0,1,0,один или два ребенка,есть дети
1,1,0,один или два ребенка,есть дети
2,0,0,нет детей,без детей
3,3,0,многодетный,есть дети
4,0,0,нет детей,без детей


In [38]:
# свожу
child_debt.pivot_table(index=['child_have'], values = ['debt'])\
.rename(columns={'debt':'share_debt'}).sort_values(by='share_debt')

,share_debt
child_have,
без детей,0.075438
есть дети,0.092403


Определенно, заемщики без детей более исправно платят кредиты, чем имеющие их.

#### **Вывод**

Наиболее добросовестными плательщиками получились заемщики имеющие пятерых детей (среди них в принципе нет должников). Однако доля таковых в имеющейся выборке ничтожна мала, потому нельзя сказать, что данные будут достоверны при увеличении выборки (но для будущей оценки таких клиентов можно поместить в категорию "многодетные"). Соответственно более достоверной будет информация, что меньше всего должников среди заемщиков, не имеющих детей (7,54% должников).

Наименнее надежные заемщики - семьи с четырьмя детьми (9,76% должников), которых также в данной выборке представлено немного, соответственно о достоверности данных, полученной на подобной выборке судить сложно, поэтому также потенциально помещаем в категорию "многодетные".

Имеющие троих детей заемщики более добросовестные плательщики (8,18% должников), чем имеющие одного или двух детей (9,23% и 9,45% должников соответственно). Семьи, имеющие одного или двух детей многочисленны (менее, чем не имеющие, однако более, чем имеющих трех и более), надежность данных категорий близка между собой, поэтому таких заемщиков также можно потенциально объединить между собой в одну категорию.

При этом хорошо видна разница между с заемщиками с детьми и без: должников среди не имеющих детей заемщиков на 1,7% меньше, чем среди тех, у кого дети имеются.

То есть зависимость между количеством детей и возвратом кредита в срок имеется, однако она не линейна:
самыми добросовестными плательщиками являются заемщики `не имеющие детей` - `7,54%` должников; затем идут `многодетные` (три и более детей) - `8,16%` должников. Наименее надежные плательщики (`9,30%` должников) - семьи с `одним либо двумя детьми`.

Вероятно такой результат вызван тем, что один либо два ребенка более распространенное количество детей в семье, чем три и более, а семья с детьми имеет больше статей расходов (куда могут быть затрачены средства, предназначенные для покрытия кредита), особенно непредвиденных, чем семья без детей.

### **Вопрос 2: Есть ли зависимость между семейным положением и возвратом кредита в срок?**

Для начала посмотрю, какие у нас имеются варианты семейного положения.

In [39]:
# уникальные варианты
data['family_status'].unique()

array(['женат / замужем', 'гражданский брак', 'вдовец / вдова',
       'в разводе', 'Не женат / не замужем'], dtype=object)

Статус `'Не женат / не замужем'`единственный, где встречается прописная буква. Для единообразия это можно было бы исправить, приведя все буквы в строчный вид, однако это было лучше делать на этапе предобработки данных. Так как сейчас мы уже находимся в исследовательской части, а рассчетам написание не мешает, данное исправление в датафрейме опущу.

Как можно было увидеть при начальном изучении датафрейма, каждому статусу в столбце `'family_status'` соответствует определенный идентификатор в графе `'family_status_id'`. Для группировки можно использовать любой из них, однако для наглядности буду использовать оба столбца.

Найду, какую долю составляют должники в зависимости от своего семейного положения. Для этого создам новую таблицу со сгруппированными по семейному статусу данными о задолженностях, а результат расчетов выведу в столбце `'share_debt'`, для удобства восприятия отсортированном по возрастанию.

In [40]:
# создаю таблицу
family_status_id_grouped = data.groupby(['family_status','family_status_id']).agg({'debt':['count', 'sum']})

In [41]:
# создаю столбец
family_status_id_grouped['share_debt'] = family_status_id_grouped['debt']['sum']/family_status_id_grouped['debt']['count']

In [42]:
# свожу
family_status_id_grouped.sort_values(by='share_debt')

debt      share_debt
                                        count  sum           
family_status         family_status_id                       
вдовец / вдова        2                   951   63   0.066246
в разводе             3                  1189   84   0.070648
женат / замужем       0                 12261  927   0.075606
гражданский брак      1                  4134  385   0.093130
Не женат / не замужем 4                  2796  273   0.097639

Хорошо видна зависимость между семейным положением и возвратом кредита в срок. Несмотря на то, что это катигориальные значения, зависимость в некотором роде линейна "жизненному пути" (благонадежность заемщика увеличивается с переходом по цепочке): "свободен"('Не женат / не замужем') -> нашел партнера ('гражданский брак') -> заключил брак ('женат / замужем') -> потерял партнера ('в разводе' и 'вдовец / вдова').

Самыми добросовестными плательщиками оказались вдовцы и вдовы (которые, однако, составляют наименьшую часть заемщиков). Самые проблемные - не женатые/не замужние граждане.

Как мы видим, большую долю заемщиков составляют женатые/замужние клиенты, что может быть связано с тем, что это более распространеннный статус, чем остальные. А также тем, что находящиеся в браке люди могут быть более склонны "вкладываться в семью и будущее", например, путем приобретения жилья.

Также в предыдущих расчетах мы увидели схожий процент просроченных платежей у незамужние и неженатые заемщиков (9,76% должников) и у лиц, состоящих в гражданском браке (9,31%), то есть тех, кто пока что "не дошел до ЗАГСа". Интересно, а что мы увидим, если эти категории объединить между собой?

In [43]:
(385+273)/(4134+2796) # считаем долю должников share_debt при объединении категорий  'гражданский брак' и 'Не женат / не замужем'

0.09494949494949495

Процент должников получился средний между этими категориями - 9,49%

А если объединить между собой тех, кто остался без партнера? То есть категории 'вдовец / вдова' и 'в разводе'.

In [44]:
(63+84)/(951+1189) # считаем долю должников share_debt при объединении категорий  'вдовец / вдова' и 'в разводе'

0.06869158878504673

Также средний между категориями процент должнков - 6,87%.

В любом случае, можно дополнительно изучить влияние наличия "документально закрепленного партнера" на возврат кредита в срок.

In [45]:
# Создаем новую таблицу с интересующими нас столбцами
family_debt = data[['family_status', 'family_status_id', 'debt']]

In [46]:
family_debt.head()  #проверяем, что создание таблицы прошло успешно

,family_status,family_status_id,debt
0,женат / замужем,0,0
1,женат / замужем,0,0
2,женат / замужем,0,0
3,женат / замужем,0,0
4,гражданский брак,1,0


In [47]:
# создам функцию
def categorize_family(family):
    try:
        if family == 0:
            return 'в официальном браке'
        else:
            return 'в браке не состоит'
    except:
        pass

In [48]:
# применяю
# ругается, но работает
family_debt['partner_have'] = family_debt['family_status_id'].apply(categorize_family)

/tmp/ipython-input-1702164026.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  family_debt['partner_have'] = family_debt['family_status_id'].apply(categorize_family)


In [49]:
family_debt.head() # проверяем, что проблемный код все-таки сработал

,family_status,family_status_id,debt,partner_have
0,женат / замужем,0,0,в официальном браке
1,женат / замужем,0,0,в официальном браке
2,женат / замужем,0,0,в официальном браке
3,женат / замужем,0,0,в официальном браке
4,гражданский брак,1,0,в браке не состоит


In [50]:
# свожу
# нас интересуют только итоги, поэтому pivot_table() достаточно
family_debt.pivot_table(index=['partner_have'], values = ['debt'])\
.rename(columns={'debt':'share_debt'}).sort_values(by='share_debt')

,share_debt
partner_have,
в официальном браке,0.075606
в браке не состоит,0.088754


Выходит, что люди, не имеющие "штампа в паспорте" в целом менее надежные заемщики, чем те, кто имеет "закрепленного ЗАГСом" партнера.

Возможно это связано с тем, что у находящихся в официальном браке людей часто общий с партнером бюджет, а долги также являются общей ответственностью (уже на уровне законодательства), и принятие решение о взятие кредита и его целей принимается обоими партнерами, соответвенно оно более взвешенное и сопоставимое с потребностями и возможностям семьи. К тому же такая "ячейка общества" более "устойчивая конструкция", имеющая определенный сопутствующий статусу набор прав и обязанностей с точки зрения законодательства, чем гражданский брак, разорвать который проще, чем официальный.

У остальных категорий (собранных в "в браке не состоит") может быть неоткуда взять дополнительные ресурсы на погашение кредита либо ввиду отсутствия постоянного партнера по каким-либо причинам, либо в связи с возможным нежеланием имеющегося партнера разделять финансовое (и иное) бремя (например, некоторая часть гражданских браков).

#### **Вывод**

Установлена зависимость своевременного возврата кредита от семейного статуса заемщика. Несмотря на то, что это катигориальные значения, зависимость в некотором роде линейна "жизненному пути" (благонадежность увеличивается с переходом по цепочке): "свободен"('Не женат / не замужем') -> нашел партнера ('гражданский брак') -> заключил брак ('женат / замужем') -> потерял партнера ('в разводе' и 'вдовец / вдова').

Самыми отвественными являются `вдовы и вдовцы` (`6,62%` должников); за ними идут лица, находящиеся `в разводе` (`7,06%`) - то есть лица из условной категории "оставшиеся без партнера" (среди данной условной категории должники составляют 6,87%).

Лица из самой многочисленной категории `женатые и замужние` занимат среднюю позицию (`7,56%`).

Наиболее склонны к просрочке платежей лица, состоящие `в гражданском браке` (`9,31%`) и `незамужние и неженатые` заемщики (`9,76%` должников), то есть лица, "без узаконенных в ЗАГСе отношений" (среди данной условной категории должники составляют 9,49%).

Близость процентов должников среди незамужних/неженатых и состоящих в гражданском браке может быть объяснена погрешностью при внесении семейного статуса (часть фактически находящихся в гражданском браке заемщиков могли указать свой статус как 'Не женат / не замужем', также его могли заявить вдовцы/вдовы и находящиеся в разводе лица, особенно если предлагалось вписывать статус вручную вместо выбора из имеющихся вариантов, что могло внести погрешности и в другие категории).

Если брать в разрезе есть "штамп в паспороте" или нет, то состоящие в официальном браке лица показывают себя более ответсвенными (7,56% должников), чем еще или уже не состоящие (8,88%). Однако это лишь дополнительная информация, а для прогнозирования лучше продолжать использовать имеющееся деление на пять статусов, так как оно хорошо отражает зависимость своевременного возврата кредита от семейного статуса заемщика, в то время как данное деление (на две категории) теряет точность за счет обобщения. При делении же на три категории ('вдовы и вдовцы' + 'в разводе', 'женатые и замужние', 'в гражданском браке' + 'незамужние и неженатые') зависимость сохраняется, однако точность оценки рисков может снизится.

### **Вопрос 3: Есть ли зависимость между уровнем дохода и возвратом кредита в срок?**

В разделе "Предобработка данных" я присвоила всем заемщикам определенную категорию в зависимости от уровня их дохода.
Категоризация была такой:

- 0–30000 — `'E'`;
- 30001–50000 — `'D'`;
- 50001–200000 — `'C'`;
- 200001–1000000 — `'B'`;
- 1000001 и выше — `'A'`.

Теперь же найду, какую долю составляют должники в зависимости от их уровня. Для этого создам новую таблицу со сгруппированными по уровню дохода данными о задолженностях, а результат расчетов выведем в столбце `'share_debt'`, для удобства восприятия отсортированном по возрастанию.

In [51]:
# создание таблицы
income_debt = data.groupby(['total_income_category']).agg({'debt':['count', 'sum']})

In [52]:
# создание столбца
income_debt['share_debt'] = income_debt['debt']['sum']/income_debt['debt']['count']

In [53]:
# сведение
income_debt.sort_values(by='share_debt')

debt       share_debt
                       count   sum           
total_income_category                        
D                        349    21   0.060172
B                       5014   354   0.070602
A                         25     2   0.080000
C                      15921  1353   0.084982
E                         22     2   0.090909

Выборки в категориях `'A'`,`'E'` и `'D'` крайне малы (особенно первые две), поэтому нельзя сказать, на сколько будут достоверны будут полученные по этим трем категориям данные при увеличении выборки. Однако ввиду своей малочисленности они также не окажут влияние на поведение больших по количеству выборок при объединении.

Для того, чтобы это проверить, объединим группы `'A'` и `'B'` в одну группу, и `'C'`, `'D'` и `'E'` - в другую. После чего повторим рассчеты.

In [54]:
(2+354)/(25+5014) # считаем долю должников в объединенной группе 'A' + 'B'

0.07064893828140505

In [55]:
(1353+21+2)/(15921+349+22) # считаем долю должников в объединенной группе 'C'+'D'+'E'

0.0844586300024552

Процент должников группы `'A' + 'B'` составляет `7,06 %` и сопоставимоставим с таковым у группы `'B'`. Аналогично процент должников группы `'C'+'D'+'E'` (`8,45%`) сопоставим с таковым у у группы `'C'`(`8,5%`).

Поэтому можно сказать, что заемщики с доходом до 200000 более склонны к просрочкам.

#### **Вывод**

Наименее добросовестными плательщиками оказались заемщики с доходами категории `'E'`(0–30000, то есть заемщики с минимальным доходом) - `9,09%` должников. Однако таких заемщиков так же мало, как и заемщиков категории `'A'`(1000001 и выше, то есть с крайне высоким уровнем дохода, но имеющих при этом целых `8%` должников), поэтому сложно сказать, насколько достоверны будут полученные по этим двум категориям данные при увеличении выборки.

Наиболее добросовестными  оказались заемщики из не слишком многочисленной (но все же большей, чем две предыдущие) категории `'D'`(30001–50000) - `6,02%` должников. Однако ввиду того, что их численность все равно заметно меньше численности оставшихся групп `'B'` и `'C'`, полученные в исследовании данные также нельзя считать достоверными при перенесении на выборку иного размера.

Поэтому в связи с малочисленностью групп `'A'`,`'D'` и `'E'` и связанной с этим незначительностью влияния на результаты  исследования для удобства оценки рисков можно объединять их с ближайшей по доходам более многочисленной группой.

Повторное исследование объединенных групп показало, что процент должников группы `'A' + 'B'`(доход более 200000) составляет `7,06 %` и сопоставимоставим с таковым у группы `'B'`(200001–1000000,`7,06%` должников). Аналогично процент должников группы `'C'+'D'+'E'` (доход менее 20000, `8,45%`) сопоставим с таковым у у группы `'C'`(50001–200000, `8,5%`).

Поэтому можно сказать, что заемщики с доходом более 200000 реже допускают просрочку.

### **Вопрос 4: Как разные цели кредита влияют на его возврат в срок?**

В разделе "Предобработка данных" я систематизировала цели взятия кредита в столбце `'purpose_category'`. Таковых получилось четыре:

- `'операции с недвижимостью'`;
- `'операции с автомобилем'`;
- `'получение образования'`;
- `'проведение свадьбы'`.

Теперь же найду, какую долю составляют должники в зависимости от целей заема. Для этого создам новую таблицу со сгруппированными по категории цели данными о задолженностях, а результат расчетов выведу в столбце `'share_debt'`, для удобства восприятия отсортированном по возрастанию.

In [56]:
# создание новой таблицы
purpose_debt = data.groupby(['purpose_category']).agg({'debt':['count', 'sum']})

In [57]:
# создание нового столбца
purpose_debt['share_debt'] = purpose_debt['debt']['sum']/purpose_debt['debt']['count']

In [58]:
# свожу
purpose_debt.sort_values(by='share_debt')

debt      share_debt
                          count  sum           
purpose_category                               
операции с недвижимостью  10751  780   0.072551
проведение свадьбы         2313  183   0.079118
получение образования      3988  369   0.092528
операции с автомобилем     4279  400   0.093480

#### **Вывод**

Обнаружена зависимость между целью кредита и его возвратом в срок.

Наиболее многочисленной категорией является `'операции с недвижимостью'`. Заемщики с данной целью также являются и наиболее надежными - доля должников составляет `7,26%`.

Следом за ней по степени добросовестности (доля должников `7,91%`) идет наименее многочисленная категория - `'проведение свадьбы'`.

Количество желающих `'получить образование'` несколько меньше, чем желающих осуществить `'операции с автомобилем'`, также и доля должников среди студентов меньше (`9,25%`), чем среди автовладельцев (`9,35%`), однако немнамного и их неблагонадежность в целом сопоставима.

### **Вопрос 5: Приведите возможные причины появления пропусков в исходных данных.**

#### **Ответ**

Пропуски в исходных данных имелись в столбцах `'days_employed'` (общий трудовой стаж в днях) и `'total_income'`(ежемесячный доход). Количество данных пропусков совпадает (2174 пропуска).

Вероятно переданый на анализ датафрейм представляет из себя компиляцию данных из нескольких источников (баз данных банка). Возможно, данные с пропусками были из одной базы (так как количества пропусков совпадают), из которой произошел либо некорректный перенос в датафрейм (графы ошибочно не были включены в выгрузку), либо изначально некорректно вносились в базу (ошибка записи в графы), либо на момент создания этой базы данные сведенья не собирались у заемщика и/или не вносились в базу.

В любом случае желательно выяснить, откуда была взята информация для датафрейма, чтобы более точно установить причины пропусков в данных и устранить их.

### **Вопрос 6: Объясните, почему заполнить пропуски медианным значением — лучшее решение для количественных переменных.**

#### **Ответ**

Удаляя строки с пропусками мы теряем данные, которые нужны нам для анализа. Поэтому лучше постараться их сохранить, а для этого необходимо заполнить пропуски характерными значениями.

Среднее значение некорректно характеризует данные, когда некоторые значения сильно выделяются среди большинства. Медиана, делящая выборку по середине по количеству элементов, а не по их сумме, в таких случаях ведет себя более корректно (а при добавлении нового элемента смещается незначительно независимо от размера нового элемента), чем позволяет принять в выборку максимальное количество данных (не отбрасывая резко отличающиеся от "основной массы"), не опасаясь искажений в выводах, и сохранить сопуствующие им в иных строках данные, которые были бы утеряны для анализа в случае исключения резко отличющихся значений (что так же могло исказить выводы исследования).

## **Общий вывод**

Для исследования была получена собранная ранее заказчиком статистика о платежеспособности клиентов.

**На этапе предобработки были выполненны следующие действия:**

1. Выявлены аномалии и прозведена их очистка:

- Отрицательное количество дней стажа. Заменены на положительные значения.

- Аномально большие медианные значения стажа для пенсионеров и безработных. Данные столбца, касающегося трудового стажа, не принимали участие в исследованиях, поэтому аномалии исправлению не подвергались.

- Отрицательное (-1) и черезмерно большое (20) количество детей. Аномалии удалены.

2. Выявлены пропуски в столбцах `'days_employed'` (общий трудовой стаж в днях) и `'total_income'`(ежемесячный доход). Для исследования пропуски в столбцах были заполненны медианными значениями для каждого типа занятости. Также значения в столбце `'total_income'` замененены на целочисленный для удобства дальнейшей работы.

3. Приведены к единству написания (только строчные буквы) данные в графе образования.

4. Произведена категоризация данных:

- Разделение заемщиков по уровню дохода на пять категорий.

- Разделение заемщиков по целям кредита.


5. Произведены обработка удаление дубликатов.


**После чего я проверила четыре гипотезы и установила:**

1. Зависимость между количеством детей и возвратом кредита в срок:

Самыми добросовестными плательщиками оказались заемщики не имеющие детей; затем идут многодетные (три и более детей). Наименее надежные плательщики - семьи с одним либо двумя детьми.

2. Зависимость между семейным положением и возвратом кредита в срок:

Заемщики состоящие или состоявшие ранее в браке реже допускают просрочку.

3. Зависимость между уровнем дохода и возвратом кредита в срок:

Заемщики с доходом более 200000 более надежные клиенты.

4. Влияние цели кредита на его возврат в срок:

Наименее рисковыми являются 'операции с недвижимостью' и 'проведение свадьбы'. Менее надежные заемщики среди желающих 'получить образование' и осуществить 'операции с автомобилем'.

**Рекомендации по улучшению сбора данных и скоринга:**

1. Выявить источник пропусков в исходных данных в столбцах `'days_employed'` (общий трудовой стаж в днях) и `'total_income'`(ежемесячный доход).

Количество данных пропусков совпадает (2174 пропуска из 21525 строк, то есть утеряно почти 10% информации).

Поэтому для улучшения качества скоринга желательно выяснить, откуда была взята информация для датафрейма, чтобы более точно установить причины пропусков (технический или человеческий фактор) в данных и устранить их. А затем можно будет повторить данное исследование для закрепления(или опровержения) текущего вывода о зависимости между уровнем дохода заемщика и его надежностью, а также проверить гипотезу о наличии зависимости между стажем заемщика и возвратом кредита в срок.

Возможно именно наличие такого количества пропусков в данных и привело к аномальным значениям стажа у безработных и пенсионеров. Но данные аномалии также могли возникнуть и по иным причинам (например, неточности при заполнении заемщиком анкеты, что тоже необходимо минимизировать).

2. Узнать причины, по которым значения дней стажа принимали отрицательные значения.

Для человеческого фактора их слишком много, к тому же в прочих столбцах таких аномалий не замечено. Так что вероятнее всего  это техническая ошибка.

3. Унифицировать способ заполнения анкеты заемщиком.

Лучше всего - давать выбор из нескольких готовых вариантов, вместо написания своего. Это облегчит процесс самому заемщику, а также снизит нагрузку на операционистов при вводе данных в систему, а также снизит вероятность ошибок при дальшнейшем анализе.

Особенно это актуально для граф "образование" и "цель кредита". Но лучше также иметь варианты в "семейном положении" и "тип занятости" с расшифровкой указанных вариантов для исключения разночтений между заемщиком и банком (например, работающий пенсионер может себя указать как сотрудник и как пенсионер, а находящийся в разводе - как "не женат").

4. Оптимизировать датафрейм как способ хранения данных о заемщиках:

- Распространить унификацию из анкеты на датафрейм.

Это позволит избежать,например, множества варианто написания одной и той же цели кредита.

- Устранить дублирование информации.

Здесь целых два столбца, отвечающих за семейное положение: `'family_status'` и `'family_status_id'`, обозначающие одно и то же, но по-разному. Для избегания дублирования информации, снижения размера датафрейма и увеличения скорости его обработки можно оставить только `'family_status_id'` и сделать отдельную сводную таблицу с расшифровкой, состоящей из `'family_status'` и `'family_status_id'`.
Также стоит поступить и со столбцами, касающимися образования заемщиков: `'education'` и `'education_id'`.

- Категоризировать цели кредита (по вышеуказанному образцу).

5. Для скоринга категоризировать заемщиков также по уровню дохода (до и более 200000) и количеству детей (нет, 1-2, 3 и более) для сокращения количества вычислений.
